# Clinical Recruitment GRPO Training — OpenEnv + TRL

This notebook trains an LLM to manage clinical trial recruitment using:
- **OpenEnv** framework for the environment
- **TRL GRPOTrainer** with `environment_factory` for RL training
- **Qwen3-0.6B** as the base model (fits on T4 GPU)

The model learns to use 8 tools (screen_patient, recontact, allocate_to_site, etc.)
to maximize patient enrollment within budget constraints.

**Runtime:** ~30 minutes on Colab T4

---

## How it works

1. TRL creates `ClinicalRecruitmentToolEnv` instances (one per generation)
2. Each instance wraps a live `ClinicalRecruitmentEnv` with 8 tool methods
3. The model generates tool calls; TRL executes them against the env
4. 5 reward functions score the episode outcome (enrollment, budget, diversity, etc.)
5. GRPO compares completions within each group to compute advantages
6. The model learns which tool-calling strategies produce better outcomes

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q "trl>=1.2.0" "datasets>=2.21.0" "accelerate>=0.34.0" "peft>=0.14.0" "openenv-core>=0.2.0" jmespath
!pip install -q pydantic fastapi httpx

# Clone the environment repo
!git clone https://github.com/pratimassaravanan/clinical-recruitment.git 2>/dev/null || echo 'Already cloned'

import sys
sys.path.insert(0, 'clinical-recruitment')

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Verify the Environment

In [ ]:
# Quick sanity check: the tool environment works
from tool_env import ClinicalRecruitmentToolEnv, REWARD_FUNCS

env = ClinicalRecruitmentToolEnv()
obs = env.reset(task="easy_bench")
print("Initial observation:")
print(obs[:200])
print()

# Try a tool call
result = env.screen_patient(patient_id="P-0001", hypothesis="noise_dominant")
print("After screen_patient:")
print(result[:200])
print(f"\nReward: {env.reward:.4f}")
print(f"Cumulative reward: {env.cumulative_reward:.4f}")
print(f"\nReward functions available: {len(REWARD_FUNCS)}")
for f in REWARD_FUNCS:
    print(f"  - {f.__name__}: {f.__doc__.strip().split(chr(10))[0]}")

## 3. Build Training Dataset

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a clinical trial recruitment agent managing patient enrollment.

Your goal is to maximize enrollment while staying within budget. You have access to tools
for screening patients, following up with candidates, allocating patients to sites, and
adjusting your recruitment strategy.

Priority rules:
1. If allocation_candidates are available and sites have capacity: use allocate_to_site
2. If recontact_candidates are available: use recontact to convert them
3. If available_patients exist: use screen_patient to start the funnel
4. Otherwise: use adjust_strategy with increase_outreach

Always provide a hypothesis about what's driving outcomes (noise_dominant, dropout_dominant, or site_bias).
Read the observation carefully and use the EXACT patient_id and site_id values shown."""

tasks = ["easy_bench", "medium_bench", "hard_bench"]
DATASET_SIZE = 150  # 50 per task

prompts = []
task_ids = []
for i in range(DATASET_SIZE):
    task = tasks[i % len(tasks)]
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"You are managing a clinical trial recruitment episode (task: {task}). "
            "Use the available tools to screen patients, follow up, allocate to sites, "
            "and adjust strategy. Maximize enrollment within the budget constraint. "
            "Read each observation carefully and choose the best action."
        )},
    ]
    prompts.append(prompt)
    task_ids.append(task)

dataset = Dataset.from_dict({"prompt": prompts, "task": task_ids})
print(f"Dataset: {len(dataset)} prompts ({DATASET_SIZE // 3} per task)")

## 4. Configure and Run GRPO Training

This is the key cell. We use `environment_factory=ClinicalRecruitmentToolEnv` which means:
- TRL creates one env instance per generation
- `reset()` is called at episode start
- Tool methods are auto-discovered and exposed to the model
- Multi-turn tool-calling loop runs until the model stops or max tokens reached
- Reward functions score the final env state

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from tool_env import (
    ClinicalRecruitmentToolEnv,
    reward_enrollment_progress,
    reward_budget_efficiency,
    reward_screening_accuracy,
    reward_action_diversity,
    reward_hypothesis_consistency,
)

MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_STEPS = 30  # Increase to 100+ for better results

reward_funcs = [
    reward_enrollment_progress,
    reward_budget_efficiency,
    reward_screening_accuracy,
    reward_action_diversity,
    reward_hypothesis_consistency,
]

grpo_config = GRPOConfig(
    output_dir="grpo_output",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_generations=4,
    max_completion_length=1024,
    logging_steps=1,
    save_steps=10,
    save_total_limit=2,
    report_to="none",
    chat_template_kwargs={"enable_thinking": False},
    log_completions=True,
)

print(f"Model: {MODEL_NAME}")
print(f"Training steps: {MAX_STEPS}")
print(f"Generations per prompt: {grpo_config.num_generations}")
print(f"Reward functions: {[f.__name__ for f in reward_funcs]}")
print(f"\nCreating trainer with environment_factory=ClinicalRecruitmentToolEnv...")

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=reward_funcs,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=ClinicalRecruitmentToolEnv,
)

print("Trainer created. Starting training...")
print("Each step: model generates tool calls -> TRL executes against live env -> rewards computed")
print()

In [ ]:
# Run training!
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Global step: {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")

## 5. Extract and Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract logged metrics from trainer state
log_history = trainer.state.log_history

# Parse out loss and reward metrics
steps = []
losses = []
rewards = {f"reward_func_{i}": [] for i in range(len(reward_funcs))}
reward_steps = {f"reward_func_{i}": [] for i in range(len(reward_funcs))}

for entry in log_history:
    if "loss" in entry and "step" in entry:
        steps.append(entry["step"])
        losses.append(entry["loss"])
    for i in range(len(reward_funcs)):
        key = f"train/reward_func_{i}"
        if key in entry and "step" in entry:
            reward_steps[f"reward_func_{i}"].append(entry["step"])
            rewards[f"reward_func_{i}"].append(entry[key])

print(f"Collected {len(steps)} loss entries and reward entries")
print(f"Loss range: {min(losses):.4f} - {max(losses):.4f}" if losses else "No loss data")

In [ ]:
# Plot 1: Training Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
if steps and losses:
    ax.plot(steps, losses, "o-", color="#2563eb", linewidth=2, markersize=4, label="Training Loss")
    # Smoothed trend
    if len(losses) > 3:
        window = min(5, len(losses))
        smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax.plot(steps[window-1:], smoothed, "--", color="#94a3b8", linewidth=1.5, label="Smoothed")
ax.set_xlabel("Training Step", fontsize=12, fontweight="bold")
ax.set_ylabel("Loss", fontsize=12, fontweight="bold")
ax.set_title("GRPO Training Loss", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Reward Functions
ax = axes[1]
reward_names = [f.__name__.replace("reward_", "") for f in reward_funcs]
colors = ["#2563eb", "#059669", "#d97706", "#7c3aed", "#dc2626"]
for i, (name, color) in enumerate(zip(reward_names, colors)):
    key = f"reward_func_{i}"
    if reward_steps[key]:
        ax.plot(reward_steps[key], rewards[key], "o-", color=color,
                linewidth=1.5, markersize=3, label=name, alpha=0.8)
ax.set_xlabel("Training Step", fontsize=12, fontweight="bold")
ax.set_ylabel("Reward", fontsize=12, fontweight="bold")
ax.set_title("Per-Component Reward Curves", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

fig.suptitle(f"GRPO Training — Clinical Recruitment ({MODEL_NAME})",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("grpo_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: grpo_training_curves.png")

## 6. Save Results

In [ ]:
import json
from datetime import datetime

results = {
    "model": MODEL_NAME,
    "method": "GRPO via TRL environment_factory",
    "environment": "ClinicalRecruitmentToolEnv",
    "max_steps": MAX_STEPS,
    "training_loss": train_result.training_loss,
    "loss_curve": losses,
    "reward_curves": {f.__name__: rewards[f"reward_func_{i}"] for i, f in enumerate(reward_funcs)},
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "timestamp": datetime.now().isoformat(),
}

with open("grpo_trl_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to grpo_trl_results.json")
print(f"\nFinal training loss: {train_result.training_loss:.4f}")
print(f"Total steps: {train_result.global_step}")

In [ ]:
# Save the trained model
trainer.save_model("grpo_trained_model")
print("Model saved to grpo_trained_model/")
print("\nDone! You can now evaluate this model against the environment.")

## 7. (Optional) Compare Trained vs Untrained

After training, you can evaluate the trained model against the environment
and compare with the baseline.

In [ ]:
# Quick comparison: run 5 episodes with the trained model
from tool_env import ClinicalRecruitmentToolEnv

print("Running 3 evaluation episodes (one per task)...\n")
for task in ["easy_bench", "medium_bench", "hard_bench"]:
    env = ClinicalRecruitmentToolEnv()
    obs = env.reset(task=task)
    # Run a few steps with the environment to check state
    for i in range(5):
        if env.done:
            break
        try:
            # Use screen_patient as a simple baseline action
            obs_dict = env.last_observation or {}
            avail = obs_dict.get("available_patients", [])
            if avail:
                result = env.screen_patient(patient_id=avail[0]["id"])
            else:
                result = env.adjust_strategy(strategy_change="increase_outreach")
        except Exception as e:
            print(f"  Error: {e}")
            break
    
    obs = env.last_observation or {}
    print(f"[{task}] enrolled={obs.get('enrolled_so_far', 0)}/{obs.get('target_enrollment', '?')} "
          f"budget={obs.get('budget_remaining', 0):.0f} "
          f"reward={env.cumulative_reward:.4f} "
          f"actions={len(env.action_history)}")